# Clasificación MNIST con PyTorch

Implementa un modelo equivalente, lo entrena y reporta accuracy de prueba. Requiere PyTorch instalado en el entorno.


# Preparación de datos MNIST

Este bloque se incluye en cada notebook para que pueda ejecutarse de forma independiente.

In [ ]:
# Librerías, semilla y carga de MNIST
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)
x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, "Dev:", x_dev.shape, "Test:", x_test.shape)


In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

class RedMNISTRegularizada(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.30)
        self.fc2 = nn.Linear(256, 10)
    def forward(self, x):
        x = self.flatten(x)
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)

def loader(x, y, batch_size=64, shuffle=False):
    xt = torch.from_numpy(x).float().unsqueeze(1)
    yt = torch.from_numpy(y.astype(np.int64))
    return DataLoader(TensorDataset(xt, yt), batch_size=batch_size, shuffle=shuffle)

train_loader = loader(x_train, y_train, shuffle=True)
dev_loader = loader(x_dev, y_dev)
test_loader = loader(x_test, y_test)
modelo_torch = RedMNISTRegularizada().to(device)
criterio = nn.CrossEntropyLoss()
optimizador_torch = torch.optim.Adam(modelo_torch.parameters(), lr=0.001, weight_decay=1e-4)
print("Dispositivo:", device)
print(modelo_torch)


In [ ]:
def evaluar(loader_datos):
    modelo_torch.eval()
    total_loss = 0.0; total = 0; correctos = 0
    with torch.no_grad():
        for xb, yb in loader_datos:
            xb, yb = xb.to(device), yb.to(device)
            logits = modelo_torch(xb)
            loss = criterio(logits, yb)
            total_loss += loss.item() * yb.size(0)
            correctos += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return total_loss / total, correctos / total

EPOCHS = 5
for epoch in range(EPOCHS):
    modelo_torch.train(); total_loss = 0.0; total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizador_torch.zero_grad()
        loss = criterio(modelo_torch(xb), yb)
        loss.backward(); optimizador_torch.step()
        total_loss += loss.item() * yb.size(0); total += yb.size(0)
    dev_loss, dev_acc = evaluar(dev_loader)
    print(f"Época {epoch+1}/{EPOCHS} - train loss: {total_loss/total:.4f} - dev loss: {dev_loss:.4f} - dev accuracy: {dev_acc:.4f}")


In [ ]:
test_loss, test_acc = evaluar(test_loader)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
